In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage
import os
from dotenv import load_dotenv
from datetime import datetime


In [5]:
load_dotenv(override=True)

True

In [6]:
# llm credentials
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = 'https://openrouter.ai/api/v1'

In [16]:
#helper functions
def get_llm(model = 'gpt-4o-mini'):
    return OpenAIChatCompletionClient(model=model, api_key=api_key, base_url=base_url)

def get_system_prompt(role, goal, backstory):
    system_prompt =f"""
    You are a ${backstory} \n\n
    Role: {role}\n\n
    Goal: {goal}
    """
    return system_prompt

def get_user_prompt(task, expected_output, **contexts):
    system_prompt = ""
    if contexts:
        for source, context in contexts.items():
            system_prompt += f"Context from {source}: {context}\n\n"
    
    system_prompt +=f"""
    Task: {task} \n\n
    Expected output: {expected_output}
    """
    return system_prompt

# save output in md file
def output_to_markdown(content, filename, folder = "output"):
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    return filepath

In [18]:
#setup agents with Auto gene
#create agents
#researcher
async def researcher_agent(idea, industry):
    model_client = get_llm()

    current_year = str(datetime.now().year)
    
    #system prompt
    role = f"Identify the market landscape, target users, and existing competitors for a startup idea. The idea: {idea}"
    goal = f"Market intelligence specialist focused on extracting factual, external context about a startup idea. The idea: {idea}"
    backstory = f"""
     Former startup research analyst who has evaluated hundreds of early-stage products across industry the idea is in. The industry: {industry}. 
    Skilled at quickly identifying patterns in market demand, user segments, and competitive positioning.
    """

    #user prompt
    task = f"""
        Analyze the startup idea and identify the target users, the problem it solves, the market need, and the existing competitors. 
        Focus on factual market context and concise insight.
        The idea: {idea}.
        The current year is {current_year}.
    """
    expected_output = f"""
        A structured. professional market research summary on {idea} startup idea containing:
        1. Startup idea summary
        2. target audience
        3. problem being solved
        4. market need
        5. 3 to 5 likely competitors or alternatives
        6. brief note on current opportunity in the market
        7. conclusion
    """

    system_prompt = get_system_prompt(role, goal, backstory)
    agent = AssistantAgent(
        'researcher',
        model_client=model_client,
        system_message=system_prompt,
        model_client_stream=True
    )
    user_prompt = get_user_prompt(task, expected_output)
    message = TextMessage(source="user", content=user_prompt)

    response = await agent.on_messages([message], cancellation_token=CancellationToken())

    return response.chat_message.content
    
#critic
async def critic_agent(idea, researcher_output):
    model_client = get_llm()
    #system prompt
    role = f"Expose weaknesses, risks, and blind spots in a start up idea. The idea: {idea}"
    goal = f"Skeptical evaluator focused on failure modes, edge cases, and market reality checks."
    backstory = f"""
     Ex-VC associate known for rejecting weak startups. Has seen many ideas fail due to poor differentiation, execution risk, or lack of real demand.
    Optimized for critical thinking over optimism.
    """
    #user prompt
    task = f"""
        Review the {idea} startup idea using the market research output. 
    Identify weaknesses, risks, hidden assumptions, execution challenges, and reasons the idea may fail in the market.
    """
    expected_output = f"""
        A concise critical analysis containing:
        1. main risks
        2. weak assumptions
        3. differentiation concerns
        4. adoption or execution challenges
        5. overall realism assessment
    """

    # call llm and pass the critic prompt
    system_prompt = get_system_prompt(role, goal, backstory)
    agent = AssistantAgent(
        'researcher',
        model_client=model_client,
        system_message=system_prompt,
        model_client_stream=True
    )

    user_prompt = get_user_prompt(task, expected_output, researcher_output = researcher_output)

    message = TextMessage(source="user", content=user_prompt)

    response = await agent.on_messages([message], cancellation_token=CancellationToken())

    return response.chat_message.content

#product_strategist
# - product_strategist
async def product_strategist_agent(idea, researcher_output, critic_output):
    model_client = get_llm()
    #system prompt
    role = f"Transform the idea into a viable, differentiated product with a clear MVP. The idea: {idea}"
    goal = f"Builder mindset focused on execution, positioning, and practical product design."
    backstory = f"""
     Product leader who has taken multiple startups from concept to launch. 
    Strong at simplifying ideas into focused MVPs and identifying the fastest path to product-market fit.
    """
    #user prompt
    task = f"""
        Using the research and critique, refine the startup idea into a sharper and more viable product concept.
    Define a focused MVP, a clearer positioning angle, and practical recommendations for improving the {idea} idea.
    """
    expected_output = f"""
        A final validation report containing:
        1. summary of the idea
        2. market opportunity snapshot
        3. key risks
        4. product strategy recommendation
        5. final verdict: Proceed, Refine, or Reject
        6. short rationale for the verdict
    """

    # call llm and pass the critic prompt
    system_prompt = get_system_prompt(role, goal, backstory)
    agent = AssistantAgent(
        'researcher',
        model_client=model_client,
        system_message=system_prompt,
        model_client_stream=True
    )

    user_prompt = get_user_prompt(task, expected_output, researcher_output = researcher_output, critic_output = critic_output)

    message = TextMessage(source="user", content=user_prompt)

    response = await agent.on_messages([message], cancellation_token=CancellationToken())

    return response.chat_message.content

# - evalsynthesizer
async def synthesizer_agent(idea, researcher_output, critic_output, product_output):
    model_client = get_llm()
    #system prompt
    role = f"Produce a clear, structured, decision-ready evaluation of the idea. The idea: {idea}"
    goal = f"Decision intelligence layer that combines all prior outputs into a coherent recommendation."
    backstory = f"""
     Strategy consultant experienced in turning fragmented insights into executive-level reports. 
    Known for clarity, structure, and actionable recommendations.
    """
    #user prompt
    task = f"""
        Present a balanced evaluation of the {idea} startup idea and provide a clear recommendation on whether to proceed, refine, or abandon the idea.
    """
    expected_output = f"""
        A final validation report containing:
        1. summary of the idea
        2. market opportunity snapshot
        3. key risks
        4. product strategy recommendation
        5. final verdict: Proceed, Refine, or Reject
        6. short rationale for the verdict
    """

    # call llm and pass the critic prompt
    system_prompt = get_system_prompt(role, goal, backstory)
    agent = AssistantAgent(
        'researcher',
        model_client=model_client,
        system_message=system_prompt,
        model_client_stream=True
    )

    user_prompt = get_user_prompt(task, expected_output, researcher_output = researcher_output, critic_output = critic_output, product_strategist=product_output)

    message = TextMessage(source="user", content=user_prompt)

    response = await agent.on_messages([message], cancellation_token=CancellationToken())

    return response.chat_message.content

In [ ]:
idea = "Ai Tutor"
research_report = await researcher_agent(idea, "Edtech")
output_to_markdown(research_report, 'research.md')

critic_output = await critic_agent(idea, research_report)
output_to_markdown(critic_output, 'critic.md')

product_report = await product_strategist_agent(idea, research_report, critic_output)
output_to_markdown(product_report, 'product_strategist.md')

judge_report = await synthesizer_agent(idea, research_report, critic_output, product_report)
output_to_markdown(judge_report, 'judge.md')